In [1]:
import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


In [2]:
class Neuron:
    def __init__(self, n_inputs, lr=0.1):
        # Вектор весов и одно смещение
        self._weights = np.random.uniform(-1, 1, size=(n_inputs,))
        self._bias = np.random.uniform(-1, 1)
        self.lr = lr

        # Кэш для обратного прохода
        self.last_input = None
        self.last_output = None

    def forward(self, x):
        """
        x: вектор входов длины n_inputs
        """
        x = np.array(x, dtype=float)
        self.last_input = x

        z = np.dot(x, self._weights) + self._bias
        y = sigmoid(z)

        self.last_output = y
        return y

    def backward(self, grad_output):
        """
        grad_output = dL/dy — градиент функции потерь по выходу нейрона.
        Возвращает dL/dx — градиент по входам.
        """

        # dy/dz = sigmoid'(z) = y * (1 - y), но y у нас уже есть (last_output)
        dy_dz = self.last_output * (1.0 - self.last_output)

        # dL/dz = dL/dy * dy/dz
        dL_dz = grad_output * dy_dz

        # Градиенты по весам и смещению
        dL_dw = self.last_input * dL_dz      # вектор
        dL_db = dL_dz                        # скаляр

        # Градиент по входу: dL/dx_i = w_i * dL/dz
        grad_input = self._weights * dL_dz

        # Обновление весов (градиентный спуск)
        self._weights -= self.lr * dL_dw
        self._bias -= self.lr * dL_db

        return grad_input


In [3]:
class Model:
    def __init__(self, lr=0.1):
        # 2 входа → 2 нейрона скрытого слоя
        self.n1 = Neuron(2, lr)
        self.n2 = Neuron(2, lr)
        # выходной нейрон принимает 2 входа (выходы скрытых нейронов)
        self.out = Neuron(2, lr)

    def forward(self, x):
        """
        x: вектор длины 2
        возвращает скаляр y_hat
        """
        h1 = self.n1.forward(x)
        h2 = self.n2.forward(x)
        hidden = np.array([h1, h2])
        y_hat = self.out.forward(hidden)
        return y_hat

    def backward(self, x, grad_output):
        """
        grad_output: dL/dy_hat — градиент по выходу модели.
        Предполагается, что forward уже был вызван и
        все нейроны хранят свои last_input/last_output.
        """

        # Сначала идём от выхода к скрытому слою
        grad_hidden = self.out.backward(grad_output)  # это вектор длины 2

        # Теперь распространяем градиент на каждый скрытый нейрон
        self.n1.backward(grad_hidden[0])
        self.n2.backward(grad_hidden[1])


In [ ]:
# ----- Данные XOR -----
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y_true = np.array([0, 1, 1, 0], dtype=float)

np.random.seed(42)

model = Model(lr=0.5)  

def train(model, epochs=10000):
    for epoch in range(epochs):
        loss_epoch = 0.0

        for x, y in zip(X, y_true):
            # 1. Прямой проход
            y_pred = model.forward(x)

            # 2. Значение loss (MSE с коэффициентом 1/2)
            loss = 0.5 * (y - y_pred) ** 2
            loss_epoch += loss

            # 3. Градиент по выходу модели (dL/dy_pred)
            grad_y = (y_pred - y)

            # 4. Обратный проход
            model.backward(x, grad_y)

        if (epoch + 1) % 1000 == 0:
            print(f"Epoch {epoch+1}, loss = {loss_epoch:.6f}")

train(model, epochs=10000)

print("\nFinal predictions:")
for x, y in zip(X, y_true):
    y_pred = model.forward(x)
    print(f"x = {x}, target = {y}, pred = {y_pred:.4f}")


Epoch 1000, loss = 0.380200
Epoch 2000, loss = 0.356206
Epoch 3000, loss = 0.019591
Epoch 4000, loss = 0.004257
Epoch 5000, loss = 0.002352
Epoch 6000, loss = 0.001616
Epoch 7000, loss = 0.001227
Epoch 8000, loss = 0.000988
Epoch 9000, loss = 0.000826
Epoch 10000, loss = 0.000709

Final predictions:
x = [0. 0.], target = 0.0, pred = 0.0205
x = [0. 1.], target = 1.0, pred = 0.9816
x = [1. 0.], target = 1.0, pred = 0.9827
x = [1. 1.], target = 0.0, pred = 0.0189
